<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/SimpleTransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Estimated run time:** ~5 minutes on T4 GPU


# Simple Transformers: Building a Sentiment Classifier from Scratch

Transformers are the architecture behind every modern large language model, from GPT to BERT to Gemini. At their core, transformers use a mechanism called **self-attention** that lets the model weigh the importance of every word relative to every other word in a sentence. This is fundamentally different from older architectures like RNNs, which process words one at a time in sequence.

The original transformer paper ("Attention Is All You Need", 2017) introduced an **encoder-decoder** architecture. The encoder reads the input and builds a rich representation; the decoder generates the output. For classification tasks like sentiment analysis, we only need the encoder half -- which is exactly what this notebook demonstrates.

In this notebook we build a transformer encoder from scratch using TensorFlow, train it on real IMDb movie reviews, and then compare our hand-built model to a pre-trained DistilBERT model loaded in a single line of code. The contrast is instructive: building from scratch teaches you what is happening inside; using a pre-trained model shows you how the industry actually works.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


### Loading and Preparing the IMDb Dataset

We start by loading the IMDb movie review dataset, which contains 25,000 training and 25,000 test reviews, each labeled as positive or negative. The reviews are already tokenized into integer sequences -- each integer maps to a word in a fixed vocabulary of 10,000 words. We pad (or truncate) every review to exactly 200 tokens so the transformer receives fixed-length inputs.

In [ ]:
# 1. Load Real Data: IMDb Movie Reviews
# We limit to 10,000 words and 200 words per review for speed
vocab_size = 10000
maxlen = 200

(x_train, y_train), (x_val, y_val) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_val = tf.keras.preprocessing.sequence.pad_sequences(x_val, maxlen=maxlen)

# 2. Define the Transformer Encoder Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=True):
        # Multi-Head Self-Attention
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output) # Residual connection

        # Feed Forward Network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output) # Residual connection

# 3. Handle Token + Positional Embedding
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions # Adding position to meaning



### The Transformer Block and Positional Embeddings

The code above defines two key components. The `TransformerBlock` implements multi-head self-attention followed by a feed-forward network, with residual connections and layer normalization -- this is the exact architecture from the original paper. The `TokenAndPositionEmbedding` layer converts each token ID into a dense vector and adds a positional embedding so the model knows the order of words. Without positional embeddings, the transformer would treat "dog bites man" and "man bites dog" identically.

In [ ]:
# 4. Build the Final Classification Model
embed_dim = 32  # Embedding size for each token
num_heads = 2   # Number of attention heads
ff_dim = 32     # Hidden layer size in feed forward network inside transformer

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x) # Summarize the sequence
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation="relu")(x)
outputs = layers.Dense(2, activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 5. Train on Real Data
print("Training Transformer on IMDb Dataset...")
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

### Building and Training the Classification Model

Here we assemble the full model: token+position embeddings feed into the transformer block, whose output is pooled into a single vector via global average pooling, then passed through a dense layer to produce a two-class (positive/negative) prediction. We train for just two epochs on the full IMDb training set. Even with this minimal configuration (32-dim embeddings, 2 attention heads), the model reaches around 87% validation accuracy. This demonstrates that the self-attention mechanism is remarkably effective even at small scale.

In [ ]:
# 4. Predict on your Specific Test Reviews
test_reviews = [
    "this movie was an absolute masterpiece with brilliant acting",
    "i hated every minute of this film the plot was a total disaster"
]

# We must use the same word index mapping used for training
word_index = tf.keras.datasets.imdb.get_word_index()

def preprocess_text(texts):
    encoded_texts = []
    for text in texts:
        # Convert words to IMDb indices
        tokens = text.lower().split()
        sequence = [word_index.get(word, 0) + 3 for word in tokens] # +3 is an IMDb dataset quirk
        encoded_texts.append(sequence)
    return tf.keras.preprocessing.sequence.pad_sequences(encoded_texts, maxlen=maxlen)

X_test = preprocess_text(test_reviews)
predictions = model.predict(X_test)

# 5. Output Results
print("\n--- Manual Transformer Results ---")
for i, review in enumerate(test_reviews):
    sentiment = "Positive" if predictions[i][1] > 0.5 else "Negative"
    print(f"Review: {review}")
    print(f"Prediction: {sentiment} ({predictions[i][0]:.4f})\n")

### Running Predictions on Custom Reviews

Now we test the trained model on two hand-crafted reviews -- one clearly positive, one clearly negative. We have to manually tokenize these reviews using the same word index that IMDb uses, which highlights an important point: the tokenization scheme must match between training and inference. The model correctly identifies the sentiment of both reviews, confirming that our from-scratch transformer has learned meaningful representations.

In [ ]:
import tensorflow as tf

# 1. Load the dataset (limiting to 10,000 most frequent words)
(x_train, y_train), _ = tf.keras.datasets.imdb.load_data(num_words=10000)

# 2. Get the word index (dictionary)
word_index = tf.keras.datasets.imdb.get_word_index()

# 3. Create a reverse word index to map integers back to words
# We shift by 3 because 0, 1, and 2 are reserved for <PAD>, <START>, and <UNK>
reverse_word_index = {value + 3: key for (key, value) in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(text_ids):
    return ' '.join([reverse_word_index.get(i, '?') for i in text_ids])

# Read and print the 5th to 10th records
# Python uses 0-based indexing, so the 5th record is index 4, and the 10th is index 9.
# The range function is exclusive of the stop value, so range(4, 10) will include indices 4, 5, 6, 7, 8, 9.
print("Querying records from 5th to 10th:")
for i in range(4, 10):
    print(f"--- Record {i+1} ---") # Displaying record number from 1-based perspective
    print(f"Label: {y_train[i]} (1 = Positive, 0 = Negative)")
    print(f"Text: {decode_review(x_train[i][:50])}...") # Printing first 50 words
    print("\n")

### Exploring the Raw Training Data

Before moving on, let us decode a few training examples back into readable English. This step is important for understanding what the model actually sees -- the `<UNK>` tokens show words that fell outside our 10,000-word vocabulary, and the `<START>` token marks the beginning of each review. Inspecting raw data is a good habit: it reveals preprocessing artifacts that can affect model performance.

In [ ]:
### Why is the label inversion happening? NOT SURE

# 1. Load the dataset (limiting to 10,000 most frequent words)
(x_train, y_train), _ = tf.keras.datasets.imdb.load_data(num_words=10000)

# 2. Get the word index (dictionary)
word_index = tf.keras.datasets.imdb.get_word_index()

# 3. Create a reverse word index to map integers back to words
# We shift by 3 because 0, 1, and 2 are reserved for <PAD>, <START>, and <UNK>
reverse_word_index = {value + 3: key for (key, value) in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(text_ids):
    return ' '.join([reverse_word_index.get(i, '?') for i in text_ids])

# 4. Read and print few sample records
# for i in range(2):
for i in range(4,10):
    print(f"--- Record {i} ---")
    print(f"Label: {y_train[i]} (1 = Positive, 0 = Negative)")
    print(f"Text: {decode_review(x_train[i][:50])}...") # Printing first 50 words
    print("\n")

### Verifying Labels Against Review Text

This cell repeats the decoding exercise with a slightly different range of records. Reading the decoded text alongside the labels helps us build intuition for what "positive" and "negative" mean in this dataset. The comment about label inversion is a useful reminder that real datasets sometimes have quirks that require investigation.

In [ ]:
### Let's do it the easier way with BERT (Note that there is no training involved here)

from transformers import pipeline

# 1. The Single-Line Model: Load a pre-trained Transformer (DistilBERT)
# This handles tokenization, encoding, and the classification head automatically.
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# 2. Test it on real data (Supervised Learning Inference)
test_reviews = [
    "This movie was an absolute masterpiece with brilliant acting.",
    "I hated every minute of this film; the plot was a total disaster."
]

results = classifier(test_reviews)

# 3. Print the results
for review, result in zip(test_reviews, results):
    print(f"Review: {review}")
    print(f"Result: {result['label']} (Confidence: {result['score']:.4f})\n")

### The Easy Way: Pre-trained DistilBERT in One Line

After building a transformer from scratch, we now load a **pre-trained** DistilBERT model using Hugging Face's `pipeline` API. This single line of code downloads a model that was trained on millions of examples and fine-tuned specifically for sentiment analysis -- no training required on our end. The confidence scores (0.9999 for positive, 0.9998 for negative) are dramatically higher than our from-scratch model, illustrating the power of transfer learning. In practice, you will almost always use pre-trained models; building from scratch is for understanding, not production.